# Petals to the Metal — Inference

Loads GoogLeNet weights from an uploaded Kaggle Dataset, runs inference,
outputs `submission.csv`.  Uses TensorFlow (pre-installed) to read TFRecords.

In [ ]:
import io
from pathlib import Path
import numpy as np
import tensorflow as tf
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}  |  PyTorch {torch.__version__}  |  TF {tf.__version__}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ── Dataset (TensorFlow TFRecord reader) ──────────────────
class PetalsDataset(Dataset):
    def __init__(self, data_dir, image_size=224, split='train', transform=None):
        if split not in ('train', 'val', 'test'):
            raise ValueError(f"split must be 'train', 'val', or 'test'")
        self.split = split
        self.transform = transform
        self.samples = []

        tfrecord_dir = Path(data_dir) / f'tfrecords-jpeg-{image_size}x{image_size}' / split
        tfrecord_paths = sorted(tfrecord_dir.glob('*.tfrec'))
        if not tfrecord_paths:
            raise FileNotFoundError(f"No .tfrec files in '{tfrecord_dir}'")

        feature_desc = {'image': tf.io.FixedLenFeature([], tf.string),
                        'class': tf.io.FixedLenFeature([], tf.int64)} \
                       if split != 'test' else \
                       {'image': tf.io.FixedLenFeature([], tf.string),
                        'id':    tf.io.FixedLenFeature([], tf.string)}

        raw_ds = tf.data.TFRecordDataset([str(p) for p in tfrecord_paths])
        for record in raw_ds:
            parsed = tf.io.parse_single_example(record, feature_desc)
            sample = {'image': parsed['image'].numpy()}
            if split == 'test':
                sample['id'] = parsed['id'].numpy()
            else:
                sample['class'] = parsed['class'].numpy()
            self.samples.append(sample)
        print(f'  {split}: {len(self.samples)} samples')

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        r = self.samples[idx]
        img = Image.open(io.BytesIO(r['image'])).convert('RGB')
        if self.transform: img = self.transform(img)
        if self.split == 'test':
            img_id = r['id'].decode('utf-8') if isinstance(r['id'], bytes) else r['id']
            return img, img_id
        return img, int(r['class'])

In [ ]:
# ── Model — MUST match googlenet.py attribute names exactly ─
class Inception(nn.Module):
    def __init__(self, in_channels, c1, c2, c3, c4):
        super().__init__()
        self.branch1 = nn.Sequential(
            nn.Conv2d(in_channels, c1, kernel_size=1),
            nn.BatchNorm2d(c1), nn.ReLU())
        self.branch2 = nn.Sequential(
            nn.Conv2d(in_channels, c2[0], kernel_size=1),
            nn.BatchNorm2d(c2[0]), nn.ReLU(),
            nn.Conv2d(c2[0], c2[1], kernel_size=3, padding=1),
            nn.BatchNorm2d(c2[1]), nn.ReLU())
        self.branch3 = nn.Sequential(
            nn.Conv2d(in_channels, c3[0], kernel_size=1),
            nn.BatchNorm2d(c3[0]), nn.ReLU(),
            nn.Conv2d(c3[0], c3[1], kernel_size=5, padding=2),
            nn.BatchNorm2d(c3[1]), nn.ReLU())
        self.branch4 = nn.Sequential(
            nn.MaxPool2d(kernel_size=3, stride=1, padding=1),
            nn.Conv2d(in_channels, c4, kernel_size=1),
            nn.BatchNorm2d(c4), nn.ReLU())

    def forward(self, x):
        return torch.cat([self.branch1(x), self.branch2(x),
                          self.branch3(x), self.branch4(x)], dim=1)


class GoogLeNet(nn.Module):
    def __init__(self, num_classes=104, dropout=0.4):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
            nn.Conv2d(64, 64, kernel_size=1),
            nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 192, kernel_size=3, padding=1),
            nn.BatchNorm2d(192), nn.ReLU(),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1))

        self.inception3a = Inception(192, 64, (96,128), (16,32), 32)
        self.inception3b = Inception(256, 128, (128,192), (32,96), 64)
        self.pool3 = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        self.inception4a = Inception(480, 192, (96,208), (16,48), 64)
        self.inception4b = Inception(512, 160, (112,224), (24,64), 64)
        self.inception4c = Inception(512, 128, (128,256), (24,64), 64)
        self.inception4d = Inception(512, 112, (144,288), (32,64), 64)
        self.inception4e = Inception(528, 256, (160,320), (32,128), 128)
        self.pool4 = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        self.inception5a = Inception(832, 256, (160,320), (32,128), 128)
        self.inception5b = Inception(832, 384, (192,384), (48,128), 128)

        self.avgpool = nn.AdaptiveAvgPool2d((1,1))
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(1024, 1024), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(1024, num_classes))

    def forward(self, x):
        x = self.stem(x)
        x = self.inception3a(x); x = self.inception3b(x); x = self.pool3(x)
        x = self.inception4a(x); x = self.inception4b(x); x = self.inception4c(x)
        x = self.inception4d(x); x = self.inception4e(x); x = self.pool4(x)
        x = self.inception5a(x); x = self.inception5b(x)
        x = self.avgpool(x); x = self.classifier(x)
        return x

In [ ]:
# ── Load weights ───────────────────────────────────────────
DATA_DIR = '/kaggle/input/tpu-getting-started'
MODEL_DIR = '/kaggle/input/YOUR_DATASET_PATH'   # <-- replace me

model = GoogLeNet(num_classes=104, dropout=0.4)
state = torch.load(f'{MODEL_DIR}/best_model.pth', map_location=device,
                   weights_only=True)
model.load_state_dict(state)
model.to(device).eval()
print(f'Params: {sum(p.numel() for p in model.parameters()):,}')
print('Weights loaded.')

In [ ]:
# ── Inference ──────────────────────────────────────────────
test_tf = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

print('Loading test data...')
test_ds = PetalsDataset(DATA_DIR, 224, 'test', test_tf)
test_iter = DataLoader(test_ds, batch_size=128, shuffle=False, num_workers=2)
print(f'Test images: {len(test_ds)}')

ids_all, preds_all = [], []
with torch.no_grad():
    for imgs, img_ids in test_iter:
        logits = model(imgs.to(device))
        preds_all.extend(logits.argmax(dim=1).cpu().tolist())
        ids_all.extend(img_ids)

print(f'Done. {len(ids_all)} predictions.')

with open('submission.csv', 'w') as f:
    f.write('id,label\n')
    for img_id, pred in zip(ids_all, preds_all):
        f.write(f'{img_id},{pred}\n')
print('submission.csv saved')